# Eksperimen Machine Learning - Iris Dataset
**Nama:** Moch Mizan Ghodafail  
**Dataset:** Iris Dataset  
**Deskripsi:** Dataset klasifikasi bunga iris berdasarkan fitur sepal dan petal  
**Tujuan:** EDA dan preprocessing data sebelum pelatihan model


## Setup Environment


In [ ]:
# Uncomment jika running di Google Colab
# !pip install scikit-learn pandas numpy matplotlib seaborn

# Jika running di Colab, clone repo terlebih dahulu:
# !git clone https://github.com/mochmizan/Eksperimen_SML_Moch-Mizan-Ghodafail.git
# %cd Eksperimen_SML_Moch-Mizan-Ghodafail/preprocessing


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries imported successfully!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')


## 2. Data Loading


In [ ]:
# Buat iris_raw.csv jika belum ada
RAW_DATA_PATH = '../iris_raw.csv'

if not os.path.exists(RAW_DATA_PATH):
    print('File iris_raw.csv belum ada, membuat dari sklearn...')
    iris_sklearn = load_iris()
    df_temp = pd.DataFrame(data=iris_sklearn.data, columns=iris_sklearn.feature_names)
    df_temp['species'] = [iris_sklearn.target_names[t] for t in iris_sklearn.target]
    df_temp.to_csv(RAW_DATA_PATH, index=False)
    print(f'iris_raw.csv berhasil dibuat di: {os.path.abspath(RAW_DATA_PATH)}')
else:
    print(f'iris_raw.csv ditemukan di: {os.path.abspath(RAW_DATA_PATH)}')


In [ ]:
# Load dataset dari file CSV
df = pd.read_csv(RAW_DATA_PATH)

print('=== Data Berhasil Dimuat ===')
print(f'Shape: {df.shape}')
print(f'Jumlah baris: {df.shape[0]}')
print(f'Jumlah kolom: {df.shape[1]}')
print()
print('5 baris pertama:')
df.head()


## 3. Exploratory Data Analysis (EDA)


In [ ]:
# Informasi umum dataset
print('=== Informasi Dataset ===')
print(df.info())
print()
print('=== Tipe Data ===')
print(df.dtypes)


In [ ]:
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
df.describe()


In [ ]:
# Cek missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Cek duplikat
print(f'\n=== Duplicate Rows ===')
duplicates = df.duplicated().sum()
print(f'Jumlah duplikat: {duplicates}')


In [ ]:
# Distribusi kelas
print('=== Distribusi Kelas (species) ===')
class_dist = df['species'].value_counts()
print(class_dist)
print(f'\nProporsi kelas:')
print(df['species'].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
class_dist.plot(kind='bar', ax=axes[0], color=['#2196F3', '#4CAF50', '#FF5722'])
axes[0].set_title('Distribusi Kelas Species', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
axes[1].pie(class_dist.values, labels=class_dist.index, autopct='%1.1f%%',
            colors=['#2196F3', '#4CAF50', '#FF5722'])
axes[1].set_title('Proporsi Kelas Species', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Plot distribusi kelas disimpan.')


In [ ]:
# Boxplot fitur per kelas
feature_cols = [c for c in df.columns if c != 'species']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    df.boxplot(column=col, by='species', ax=axes[i])
    axes[i].set_title(f'{col}', fontsize=12)
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel('Value')

plt.suptitle('Distribusi Fitur per Kelas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Boxplot fitur per kelas disimpan.')


In [ ]:
# Heatmap korelasi
numeric_df = df[feature_cols]
corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Heatmap Korelasi Antar Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

print('Korelasi tertinggi:')
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0].drop_duplicates()
print(corr_pairs.head(5))


In [ ]:
# Histogram distribusi setiap fitur
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

colors = {'setosa': '#2196F3', 'versicolor': '#4CAF50', 'virginica': '#FF5722'}

for i, col in enumerate(feature_cols):
    for species, color in colors.items():
        data = df[df['species'] == species][col]
        axes[i].hist(data, alpha=0.6, label=species, color=color, bins=20)
    axes[i].set_title(f'Distribusi: {col}', fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()

plt.suptitle('Histogram Distribusi Fitur per Spesies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_histogram.png', dpi=100, bbox_inches='tight')
plt.show()
print('Histogram distribusi fitur disimpan.')


## 4. Data Preprocessing


In [ ]:
# Buat salinan dataframe untuk preprocessing
df_processed = df.copy()
print('Salinan dataframe dibuat untuk preprocessing.')
print(f'Shape awal: {df_processed.shape}')


In [ ]:
# Handle missing values (tidak ada, tapi sebagai template)
print('=== Handling Missing Values ===')
print(f'Missing values sebelum: {df_processed.isnull().sum().sum()}')

# Jika ada missing values pada fitur numerik, isi dengan median
for col in feature_cols:
    if df_processed[col].isnull().any():
        median_val = df_processed[col].median()
        df_processed[col].fillna(median_val, inplace=True)
        print(f'  {col}: diisi dengan median = {median_val:.4f}')

print(f'Missing values sesudah: {df_processed.isnull().sum().sum()}')


In [ ]:
# Handle duplicates
print('=== Handling Duplicates ===')
print(f'Jumlah duplikat sebelum: {df_processed.duplicated().sum()}')
df_processed = df_processed.drop_duplicates().reset_index(drop=True)
print(f'Jumlah duplikat sesudah: {df_processed.duplicated().sum()}')
print(f'Shape setelah drop duplicates: {df_processed.shape}')


In [ ]:
# Label Encoding target
print('=== Label Encoding ===')
le = LabelEncoder()
df_processed['species'] = le.fit_transform(df_processed['species'])

print('Mapping label encoding:')
for idx, cls in enumerate(le.classes_):
    print(f'  {cls} -> {idx}')

print(f'\nDistribusi setelah encoding:')
print(df_processed['species'].value_counts().sort_index())


In [ ]:
# Pisahkan features dan target
X = df_processed[feature_cols]
y = df_processed['species']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Fitur yang digunakan: {feature_cols}')

# Feature Scaling menggunakan StandardScaler
print('\n=== Feature Scaling (StandardScaler) ===')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print('Statistik setelah scaling (mean harus mendekati 0, std mendekati 1):')
print(X_scaled.describe().loc[['mean', 'std']].round(4))


In [ ]:
# Train-Test Split
print('=== Train-Test Split ===')
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')
print(f'\nProporsi split: {len(X_test)/len(X_scaled)*100:.1f}% test')

print('\nDistribusi kelas train:')
print(y_train.value_counts().sort_index())
print('\nDistribusi kelas test:')
print(y_test.value_counts().sort_index())


In [ ]:
# Simpan hasil preprocessing
OUTPUT_DIR = 'iris_preprocessing'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gabungkan X dan y lalu simpan
train_df = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
)
test_df = pd.concat(
    [X_test.reset_index(drop=True), y_test.reset_index(drop=True)],
    axis=1
)

train_path = os.path.join(OUTPUT_DIR, 'train.csv')
test_path  = os.path.join(OUTPUT_DIR, 'test.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print('=== Data Berhasil Disimpan ===')
print(f'Train data: {train_path} | Shape: {train_df.shape}')
print(f'Test data:  {test_path} | Shape: {test_df.shape}')
print(f'\nKolom output: {list(train_df.columns)}')


In [ ]:
# Verifikasi hasil
print('=== Verifikasi Hasil Preprocessing ===')
train_check = pd.read_csv(train_path)
test_check  = pd.read_csv(test_path)

print(f'Train CSV shape: {train_check.shape}')
print(f'Test CSV shape:  {test_check.shape}')
print(f'\nKolom: {list(train_check.columns)}')
print(f'\nSample train data:')
print(train_check.head(3))
print(f'\nMissing values train: {train_check.isnull().sum().sum()}')
print(f'Missing values test:  {test_check.isnull().sum().sum()}')
print('\nPreprocessing selesai!')
